In [1]:
from config import *

In [ ]:
class Image_charge_calculator:

    ### Following ELECTRIC FIELDS IN DIELECTRIC MULTI-LAYERS CALCULATED BY DIGITAL COMPUTER by Takeshi Takashima an, Ryozo Ishibashi
    ### IEEE Trans. Electr. Insul, Vol EI-13, No 1, February 1978 

    ###################################################################################################################################################

    def __init__(self, q_initial, x_initial, ep1, ep2, ep3, b, c, delta):
        
        self.q_initial = q_initial  # Initial charge q_1i
        self.x_initial = x_initial # position of initial charge q 
        
        self.ep1 = ep1  # dielectric constant 1 (rightmost layer)
        self.ep2 = ep2  # dielectric constant 2 (middle layer)
        self.ep3 = ep3  # dielectric constant 3 (leftmost layer, has conductor to the left of it) 
        #self.a = a  # thickness dielectric 1
        self.b = b  # thickness dielectric 2
        self.c = c  # thickness dielectric 3
        self.delta = delta  # Small positive number for termination condition

        self.image_charges_q5s = []  # Store generated image charges
        self.image_charges_q4s = []  # Store generated image charges
        self.image_charges_q3s = []  # Store generated image charges
        self.image_charges_q2s = []  # Store generated image charges
        self.image_charges_q1s = []  # Store generated image charges

        self.alpha12 = ( ep1 - ep2 )/ (ep1 + ep2)
        self.beta12 = 1- self.alpha12

        self.alpha21 = -self.alpha12
        self.beta21 = 1 - self.alpha21

        self.alpha23 = ( ep2 - ep3 )/ (ep2 + ep3)
        self.beta23 = 1 - self.alpha23

        self.alpha32 = - self.alpha23 
        self.beta32 = 1 - self.alpha32

        #self.q0 = self.alpha12* self.q_initial
        #self.q11 = self.beta12 * self.q_initial

    def rule_i(self, q1i, x1i, level):

        if abs(q1i / self.q_initial) < self.delta:
            return  # Termination condition

        q2j = self.alpha23 * q1i
        x2j =  - x1i + 2* self.c
        q3k = self.beta23 * q1i
        x3k =  x1i
        
        self.image_charges_q2s.append((q2j, x2j, level, 'q2j'))
        self.image_charges_q3s.append((q3k, x3k, level, 'q3k'))
        
        self.rule_ii(q2j, x2j, level + 1)
        self.rule_iii(q3k, x3k, level + 1)

    def rule_ii(self, q2i, x2i, level):

        if abs(q2i / self.q_initial) < self.delta:
            return  # Termination condition
        
        q1j = self.alpha21 * q2i
        x1j = 2*(self.b + self.c) - x2i
        q5k = self.beta21 * q2i
        x5k = x2i
        
        self.image_charges_q1s.append((q1j, x1j, level, 'q1j'))
        self.image_charges_q5s.append((q5k, x5k, level, 'q5k'))

        #print("image charge = ", q5k , "co-ordinate = ", x5k, "level =", level, "\n")
        
        self.rule_i(q1j, x1j, level + 1)

    def rule_iii(self, q3i, x3i, level):

        if abs(q3i / self.q_initial) < self.delta:
            return  # Termination condition

        q4j = -1 * q3i
        x4j = -x3i
        
        self.image_charges_q4s.append((q4j, x4j, level, 'q4j'))
        self.rule_iv(q4j, x4j, level + 1)

    def rule_iv(self, q4i, x4i, level):

        if abs(q4i / self.q_initial) < self.delta:
            return  # Termination condition

        q3j = self.alpha32 * q4i
        x3j = -x4i + 2*self.c
        q2k = self.beta32 * q4i
        x2k = x4i
        
        self.image_charges_q3s.append((q3j, x3j, level, 'q3j'))
        self.image_charges_q2s.append((q2k, x2k, level, 'q2k'))
        
        self.rule_iii(q3j, x3j, level + 1)
        self.rule_ii(q2k, x2k, level + 1)

In [3]:
class Charge_in_medium1(Image_charge_calculator):

    def calc_image_charges(self):

        q11 = self.beta12*self.q_initial
        x11 = self.x_initial
        level = 0

        self.image_charges_q1s.append((q11, x11, level, 'q1j'))
        self.rule_i(q11, x11, level + 1)

        return len(self.image_charges_q5s)


    def calc_potential_medium1(self, x, y):

        pot = self.q_initial/math.sqrt((x-self.x_initial)**2+(y)**2)

        q1 = self.alpha12*self.q_initial
        x1 = -self.x_initial + 2*(self.b+self.c)

        pot = pot + q1/math.sqrt((x-x1)**2+(y)**2)

        for image_chrg in self.image_charges_q5s:

            pot = pot+ image_chrg[0]/ math.sqrt((x-image_chrg[1])**2+(y)**2)


        return 1/4/math.pi/(8.85e-12)/self.ep1*pot

    def calc_potential_medium2(self,x,y):

        pot=0 

        for image_chrg in self.image_charges_q1s:

            pot = pot+ image_chrg[0]/ math.sqrt((x-image_chrg[1])**2+(y)**2)

        for image_chrg in self.image_charges_q2s:

            pot = pot+ image_chrg[0]/ math.sqrt((x-image_chrg[1])**2+(y)**2)


        return 1/4/math.pi/(8.85e-12)/self.ep2*pot
    
    def calc_potential_medium3(self,x,y):

        pot=0 

        for image_chrg in self.image_charges_q3s:

            pot = pot+ image_chrg[0]/ math.sqrt((x-image_chrg[1])**2+(y)**2)

        for image_chrg in self.image_charges_q4s:

            pot = pot+ image_chrg[0]/ math.sqrt((x-image_chrg[1])**2+(y)**2)


        return 1/4/math.pi/(8.85e-12)/self.ep3*pot
    
    def calc_disp_medium1(self, x, y):

        q1 = self.alpha12*self.q_initial
        x1 = -self.x_initial + 2*(self.b+self.c)

        ele = self.q_initial*(x-self.x_initial)/((x-self.x_initial)**2+y**2)**(3/2)
        ele = ele+ q1*(x-x1)/((x-x1)**2+y**2)**(3/2)

        for ic in self.image_charges_q5s:

            ele = ele+ ic[0]*(x-ic[1])/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)*ele
    
    def calc_disp_medium2(self, x, y):

        ele=0

        for ic in self.image_charges_q1s:

            ele = ele+ ic[0]*(x-ic[1])/((x-ic[1])**2+y**2)**(3/2)

        for ic in self.image_charges_q2s:

            ele = ele+ ic[0]*(x-ic[1])/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)*ele
    
    def calc_disp_medium3(self, x, y):

        ele=0

        for ic in self.image_charges_q3s:

            ele = ele+ ic[0]*(x-ic[1])/((x-ic[1])**2+y**2)**(3/2)

        for ic in self.image_charges_q4s:

            ele = ele+ ic[0]*(x-ic[1])/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)*ele
    
    
    def calc_ele_parallel1(self, x, y):

        ele=0

        q1 = self.alpha12*self.q_initial
        x1 = -self.x_initial + 2*(self.b+self.c)

        ele = self.q_initial*(y)/((x-self.x_initial)**2+y**2)**(3/2)
        ele = ele+ q1*(y)/((x-x1)**2+y**2)**(3/2)

        for ic in self.image_charges_q5s:

            ele = ele+ ic[0]*(y)/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)/self.ep1*ele
    
    def calc_ele_parallel2(self, x, y):

        ele=0

        for ic in self.image_charges_q1s:

            ele = ele+ ic[0]*(y)/((x-ic[1])**2+y**2)**(3/2)

        for ic in self.image_charges_q2s:

            ele = ele+ ic[0]*(y)/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)/self.ep2*ele
    
    def calc_ele_parallel3(self, x, y):

        ele=0

        for ic in self.image_charges_q3s:

            ele = ele+ ic[0]*(y)/((x-ic[1])**2+y**2)**(3/2)

        for ic in self.image_charges_q4s:

            ele = ele+ ic[0]*(y)/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)/self.ep3*ele
    

In [4]:
class Charge_in_medium3(Charge_in_medium1):

    def calc_image_charges(self):

        q40 = self.q_initial
        x40 = self.x_initial
        level = 0

        q41 = -self.q_initial
        x41 = -self.x_initial
        level = 0

        self.image_charges_q4s.append((q40, x40, level, 'q4j'))
        self.image_charges_q4s.append((q41, x41, level, 'q4j'))
        self.rule_iv(q40, x40, level + 1)
        self.rule_iv(q41, x41, level + 1)

        return len(self.image_charges_q5s)
    
    def calc_potential_medium1(self, x, y):

        pot = 0

        for image_chrg in self.image_charges_q5s:

            pot = pot+ image_chrg[0]/ math.sqrt((x-image_chrg[1])**2+(y)**2)


        return 1/4/math.pi/(8.85e-12)/self.ep1*pot
    
    def calc_disp_medium1(self, x, y):

        ele = 0

        for ic in self.image_charges_q5s:

            ele = ele+ ic[0]*(x-ic[1])/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)*ele
    
    def calc_ele_parallel1(self, x, y):

        ele = 0

        for ic in self.image_charges_q5s:

            ele = ele+ ic[0]*(y)/((x-ic[1])**2+y**2)**(3/2)

        return 1/4/math.pi/(8.85e-12)/self.ep1*ele
    

In [13]:
Q_initial = 1e-12 ### 1pC in C

X_initial = 5e-2 #### cm 

C = 2e-2
B = 1e-2

EP3 = 1
EP2 = 4
EP1 = 1

het = Charge_in_medium1(Q_initial, X_initial, EP1, EP2, EP3, B, C, 1e-4)

Img_chrgs = het.calc_image_charges() 

print(Img_chrgs)

X = 3E-2
Y = 0


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 3E-2
Y = 2E-2


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 3E-2
Y = 6E-2


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 0


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 2E-2


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 6E-2


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)


X = 0E-2
Y = 1E-2


V_in_medium3 = het.calc_potential_medium3(X, Y)
#V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3)

X = 100
Y = 1


V_in_medium1 = het.calc_potential_medium1(X, Y)
#V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1)


71471
x= 0.03 , y= 0 , potential in medium 1 = 0.1951563055887999 , potential in medium 2 = 0.1951492505944171
x= 0.03 , y= 0.02 , potential in medium 1 = 0.1344305111244297 , potential in medium 2 = 0.1344234551789837
x= 0.03 , y= 0.06 , potential in medium 1 = 0.04265620476947504 , potential in medium 2 = 0.042649142786185215
x= 0.02 , y= 0 , potential in medium 3 = 0.14430768669849325 , potential in medium 2 = 0.14453414515748458
x= 0.02 , y= 0.02 , potential in medium 3 = 0.11133073614166408 , potential in medium 2 = 0.11155708260760389
x= 0.02 , y= 0.06 , potential in medium 3 = 0.03856537313369244 , potential in medium 2 = 0.038790829076703816
x= 0.0 , y= 0.01 , potential in medium 3 = 0.0003885364642591926
x= 100 , y= 1 , potential in medium 1 = 3.6210731924350947e-06


In [33]:
X = 3E-2
Y = 0


D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)


X = 3E-2
Y = 2E-2

D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)


X = 3E-2
Y = 6E-2

D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)

X = 2E-2
Y = 0E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium3,", disp in medium 3 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium3,", Ey in medium 3 =", Ey_in_medium2)

X = 2E-2
Y = 2E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium3,", disp in medium 3 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium3,", Ey in medium 3 =", Ey_in_medium2)

X = 2E-2
Y = 6E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium3,", disp in medium 3 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium3,", Ey in medium 3 =", Ey_in_medium2)

x= 0.03 , y= 0 , disp in medium 1 = -34.468734952801974 , disp in medium 2 = -34.4687620122131
x= 0.03 , y= 0 , Ey in medium 1 = 0.0 , Ey in medium 2 = 0.0
x= 0.03 , y= 0.02 , disp in medium 1 = -11.813108585709307 , disp in medium 2 = -11.813135813879384
x= 0.03 , y= 0.02 , Ey in medium 1 = 3.8447636754210275 , Ey in medium 2 = 3.8447637681986064
x= 0.03 , y= 0.06 , disp in medium 1 = -1.236157200903518 , disp in medium 2 = -1.2361856331793768
x= 0.03 , y= 0.06 , Ey in medium 1 = 1.1457013411292032 , Ey in medium 2 = 1.145701518489047
x= 0.02 , y= 0.0 , disp in medium 1 = -9.54855436206075 , disp in medium 3 = -9.549982405126972
x= 0.02 , y= 0.0 , Ey in medium 1 = 0.0 , Ey in medium 3 = 0.0
x= 0.02 , y= 0.02 , disp in medium 1 = -6.423022655960876 , disp in medium 3 = -6.424448627767725
x= 0.02 , y= 0.02 , Ey in medium 1 = 2.5093654512814343 , Ey in medium 3 = 2.5093766430619637
x= 0.02 , y= 0.06 , disp in medium 1 = -1.8265100241796004 , disp in medium 3 = -1.8279195577178167
x= 0.02

In [6]:
Q_initial = 1e-12 ### 1pC in C

X_initial = 1e-2 #### cm 

C = 2e-2
B = 1e-2

EP3 = 1
EP2 = 4
EP1 = 1

het = Charge_in_medium3(Q_initial, X_initial, EP1, EP2, EP3, B, C, 1e-4)

Img_chrgs = het.calc_image_charges() 

print(Img_chrgs)

X = 3E-2
Y = 0


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 3E-2
Y = 2E-2


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 3E-2
Y = 6E-2


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 0


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 2E-2


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 6E-2


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)





176906
x= 0.03 , y= 0 , potential in medium 1 = 0.1747013423234169 , potential in medium 2 = 0.174701457450142
x= 0.03 , y= 0.02 , potential in medium 1 = 0.10006289144720111 , potential in medium 2 = 0.1000630062815036
x= 0.03 , y= 0.06 , potential in medium 1 = 0.020196792970941797 , potential in medium 2 = 0.020196905502863384
x= 0.02 , y= 0 , potential in medium 3 = 0.3060648500838602 , potential in medium 2 = 0.30606775697272287
x= 0.02 , y= 0.02 , potential in medium 3 = 0.10974482112824821 , potential in medium 2 = 0.10974772245432307
x= 0.02 , y= 0.06 , potential in medium 3 = 0.019053656019650197 , potential in medium 2 = 0.0190565140392008


In [8]:
X = 3E-2
Y = 0


D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)


X = 3E-2
Y = 2E-2

D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)


X = 3E-2
Y = 6E-2

D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)

X = 2E-2
Y = 0E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 3 =", D_in_medium3,", disp in medium 3 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 3 =", Ey_in_medium3,", Ey in medium 3 =", Ey_in_medium2)

X = 2E-2
Y = 2E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 3 =", D_in_medium3,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 3 =", Ey_in_medium3,", Ey in medium 2 =", Ey_in_medium2)

X = 2E-2
Y = 6E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 3 =", D_in_medium3,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 3 =", Ey_in_medium3,", Ey in medium 2 =", Ey_in_medium2)

x= 0.03 , y= 0 , disp in medium 1 = 11.953549882547362 , disp in medium 2 = 11.953551754742458
x= 0.03 , y= 0 , Ey in medium 1 = 0.0 , Ey in medium 2 = 0.0
x= 0.03 , y= 0.02 , disp in medium 1 = 3.284920485091224 , disp in medium 2 = 3.2849223476386937
x= 0.03 , y= 0.02 , Ey in medium 1 = 4.265214312466099 , Ey in medium 2 = 4.265214341686725
x= 0.03 , y= 0.06 , disp in medium 1 = -0.09621621075075466 , disp in medium 2 = -0.09621442485449817
x= 0.03 , y= 0.06 , Ey in medium 1 = 0.7056115196001338 , Ey in medium 2 = 0.7056116046058726
x= 0.02 , y= 0.0 , disp in medium 3 = 126.22492289037795 , disp in medium 3 = 126.22488308486328
x= 0.02 , y= 0.0 , Ey in medium 3 = 0.0 , Ey in medium 3 = 0.0
x= 0.02 , y= 0.02 , disp in medium 3 = 3.038724286411286 , disp in medium 2 = 3.0386846774877387
x= 0.02 , y= 0.02 , Ey in medium 3 = 6.212048858979356 , Ey in medium 2 = 6.2120494133756
x= 0.02 , y= 0.06 , disp in medium 3 = -0.8179538465750154 , disp in medium 2 = -0.8179919790122865
x= 0.02 , y=

In [9]:
Q_initial = 1e-12 ### 1pC in C

X_initial = 1e-2 #### cm 

C = 2e-2
B = 1e-2

EP3 = 1
EP2 = 1
EP1 = 1

het = Charge_in_medium3(Q_initial, X_initial, EP1, EP2, EP3, B, C, 1e-4)

Img_chrgs = het.calc_image_charges() 

print(Img_chrgs)

X = 3E-2
Y = 0


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 3E-2
Y = 2E-2


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 3E-2
Y = 6E-2


V_in_medium1 = het.calc_potential_medium1(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 1 =", V_in_medium1,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 0


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 2E-2


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)

X = 2E-2
Y = 6E-2


V_in_medium3 = het.calc_potential_medium3(X, Y)
V_in_medium2 = het.calc_potential_medium2(X, Y)

print("x=", X,", y=", Y, ", potential in medium 3 =", V_in_medium3,", potential in medium 2 =", V_in_medium2)





2
x= 0.03 , y= 0 , potential in medium 1 = 0.22479511736143418 , potential in medium 2 = 0.22479511736143418
x= 0.03 , y= 0.02 , potential in medium 1 = 0.11684543835570778 , potential in medium 2 = 0.11684543835570778
x= 0.03 , y= 0.06 , potential in medium 1 = 0.017479019838537502 , potential in medium 2 = 0.017479019838537502
x= 0.02 , y= 0 , potential in medium 3 = 0.5994536462971575 , potential in medium 2 = 0.5994536462971575
x= 0.02 , y= 0.02 , potential in medium 3 = 0.152737939322446 , potential in medium 2 = 0.152737939322446
x= 0.02 , y= 0.06 , potential in medium 3 = 0.013782448333578869 , potential in medium 2 = 0.013782448333578869


In [15]:

X = 3E-2
Y = 6E-2


Q_initial/4/math.pi/(8.85e-12)/EP1/math.sqrt((X-X_initial)**2+(Y)**2) - Q_initial/4/math.pi/(8.85e-12)/EP1/math.sqrt((X+X_initial)**2+(Y)**2) 

0.017479019838537516

In [16]:
X = 3E-2
Y = 0


D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)


X = 3E-2
Y = 2E-2

D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)


X = 3E-2
Y = 6E-2

D_in_medium1 = het.calc_disp_medium1(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium1 = het.calc_ele_parallel1(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 1 =", D_in_medium1,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 1 =", Ey_in_medium1,", Ey in medium 2 =", Ey_in_medium2)

X = 2E-2
Y = 0E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 3 =", D_in_medium3,", disp in medium 3 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 3 =", Ey_in_medium3,", Ey in medium 3 =", Ey_in_medium2)

X = 2E-2
Y = 2E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 3 =", D_in_medium3,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 3 =", Ey_in_medium3,", Ey in medium 2 =", Ey_in_medium2)

X = 2E-2
Y = 6E-2

D_in_medium3 = het.calc_disp_medium3(X, Y)
D_in_medium2 = het.calc_disp_medium2(X, Y)

Ey_in_medium3 = het.calc_ele_parallel3(X, Y)
Ey_in_medium2 = het.calc_ele_parallel2(X, Y)

print("x=", X,", y=", Y, ", disp in medium 3 =", D_in_medium3,", disp in medium 2 =", D_in_medium2)
print("x=", X,", y=", Y, ", Ey in medium 3 =", Ey_in_medium3,", Ey in medium 2 =", Ey_in_medium2)

x= 0.03 , y= 0 , disp in medium 1 = 16.859633802107567 , disp in medium 2 = 16.859633802107567
x= 0.03 , y= 0 , Ey in medium 1 = 0.0 , Ey in medium 2 = 0.0
x= 0.03 , y= 0.02 , disp in medium 1 = 3.9264502857531127 , disp in medium 2 = 3.9264502857531127
x= 0.03 , y= 0.02 , Ey in medium 1 = 5.937078939473954 , Ey in medium 2 = 5.937078939473954
x= 0.03 , y= 0.06 , disp in medium 1 = -0.24831923541347664 , disp in medium 2 = -0.24831923541347664
x= 0.03 , y= 0.06 , Ey in medium 1 = 0.6938180135002732 , Ey in medium 2 = 0.6938180135002732
x= 0.02 , y= 0.0 , disp in medium 3 = 79.92715283962099 , disp in medium 3 = 79.92715283962099
x= 0.02 , y= 0.0 , Ey in medium 3 = 0.0 , Ey in medium 3 = 0.0
x= 0.02 , y= 0.02 , disp in medium 3 = 2.2874117359205464 , disp in medium 2 = 2.2874117359205464
x= 0.02 , y= 0.02 , Ey in medium 3 = 12.248293977124838 , Ey in medium 2 = 12.248293977124838
x= 0.02 , y= 0.06 , disp in medium 3 = -0.4940874415231618 , disp in medium 2 = -0.4940874415231618
x= 0.02 

In [25]:
X = 2E-2
Y = 0E-2


print(Q_initial/4/math.pi/(8.85E-12)*(X-X_initial)/((X-X_initial)**2+(Y)**2)**1.5 - Q_initial*(X+X_initial)/4/math.pi/(8.85E-12)/((X+X_initial)**2+(Y)**2)**1.5 )

Q_initial/4/math.pi/(8.85E-12)*(Y)/((X-X_initial)**2+(Y)**2)**1.5 -Q_initial*(Y)/4/math.pi/(8.85E-12)/((X+X_initial)**2+(Y)**2)**1.5 

79.92715283962099


0.0